# ຝຶກ Lao ASR (pruned) ເທິງ Google Colab

**notebook 2026-09-26c** — ຖ້າບໍ່ເຫັນເລກນີ້ໃນຜົນຂອງ cell ທຳອິດ ແປວ່າເຈົ້າກຳລັງໃຊ້ notebook ເກົ່າ.

**ວິທີໃຊ້**
1. `Runtime` → `Change runtime type` → **T4 GPU** (ໃຊ້ຟຣີ)
2. `Runtime` → `Run all` (ຫຼື ກົດ ▶ ແຕ່ລະ cell)
3. ລໍຖ້າ ~15–30 ນາທີ (dataset 1.2 GB + 3 epochs ເທິງ T4)
4. Cell ສຸດທ້າຍຈະດາວໂຫຼດ `laoasr-p8ft-int8.onnx` (~152 MB) ລົງຄອມ

**ເປົ້າໝາຍ**: ຕັດ model xls-r-300m ຈາກ 24 → 8 encoder layers ເພື່ອໃຫ້ຢູ່ໃນລົດໄດ້
(ລົດມີ lowmemorykiller ຈຳກັດແຮັມຕໍ່ process) ແລ້ວ fine-tune ຄືນໃຫ້ຄວາມໜູນກັບມາ.

**ຜົນທີ່ຕ້ອງໄດ້**: eval CER ຫຼຸດຈາກ 79.5% (ບໍ່ຝຶກ) → ຄາດວ່າ < 15%.

> notebook ຈະດຶງ `finetune-pruned.py` ສະບັບໃໝ່ສຸດຈາກ GitHub ໃນ cell ຕໍ່ໄປ ⇒ ບໍ່ຕ້ອງອັບເດດ notebook ອີກ.

In [ ]:
!nvidia-smi -L || echo 'no GPU - switch the runtime to T4 GPU'
# protobuf pinned: Colab's preinstalled packages (grpcio-status, ydf, google-ai-*) need <6
!pip install -q 'transformers>=4.40' datasets soundfile pyarrow onnx onnxruntime huggingface_hub 'protobuf>=5.29.1,<6'
# ຄຳເຕືອນ 'dependency conflicts' ຂອງ pip ບໍ່ເປັນຫຍັງ ຖ້າ protobuf ຍັງ <6

In [ ]:
print('notebook 2026-09-26c')  # ຖ້າບໍ່ເຫັນເລກນີ້ = notebook ເກົ່າ
# ດຶງສະຄຣິບໃໝ່ສຸດທຸກຄັ້ງ (ບໍ່ຝັງສະຄຣິບໃນ notebook ເພື່ອກັນສະບັບເກົ່າຄ້າງ)
!wget -q -O finetune-pruned.py https://raw.githubusercontent.com/toumoua/touex-modules/main/tools/finetune-pruned.py
import hashlib
print('script sha256', hashlib.sha256(open('finetune-pruned.py','rb').read()).hexdigest()[:16],
      '| version marker:', [l for l in open('finetune-pruned.py', encoding='utf-8') if l.startswith('VERSION')][0].strip())

In [ ]:
# batch 8 + accum 2 = effective 16.  T4 (15 GB) runs out of memory at batch 16 x 20 s.
# ສຳລັບການທົດລອງໄວ: --limit 800 --epochs 1
!python finetune-pruned.py --layers 8 --epochs 3 --batch 8 --accum 2 --lr 1e-4 --eval-n 40 --max-sec 20 --export p8ft

In [ ]:
# ກວດ int8 model ດ້ວຍ 5 ປະໂຫຍກທົດສອບ: ພິມ CER + ຕົວຢ່າງ REF/HYP
import io, numpy as np, soundfile as sf, onnxruntime as ort, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

tok = {}
for line in open('model/vocab.txt', encoding='utf-8'):
    i, t = line.rstrip('\n').split('\t', 1)
    tok[int(i)] = t
mdl = ort.InferenceSession('model/laoasr-p8ft-int8.onnx', providers=['CPUExecutionProvider'])

def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

p = hf_hub_download('SiangLao/lao-asr-thesis-dataset',
                    'data/test-00000-of-00001.parquet', repo_type='dataset')
err = tot = 0
for row in pq.read_table(p).slice(0, 5).to_pylist():
    wav, sr = sf.read(io.BytesIO(row['audio']['bytes']), dtype='float32')
    wav = (wav - wav.mean()) / np.sqrt(wav.var() + 1e-7)
    out = mdl.run(None, {'input_values': wav[None, :].astype(np.float32)})[0]
    ids = out[0].argmax(-1)
    ded, prev = [], -1
    for k in ids:
        if k != prev and k != 0:
            ded.append(int(k))
        prev = k
    hyp = ''.join(tok.get(i, '') for i in ded)
    ref = ''.join(row['text'].split())
    h = ''.join(hyp.split())
    err += lev(ref, h)
    tot += max(len(ref), 1)
    print('REF', row['text'][:70])
    print('HYP', hyp[:70])
print(f'CER = {100*err/max(tot,1):.1f}%  (target: < 15%)')

In [ ]:
from google.colab import files
import os
for f in ('model/laoasr-p8ft-int8.onnx', 'model/laoasr-p8ft-fp32.onnx'):
    print(f, round(os.path.getsize(f) / 1e6, 1), 'MB') if os.path.exists(f) else print('missing', f)
files.download('model/laoasr-p8ft-int8.onnx')
# ຖ້າດາວໂຫຼດບໍ່ໄດ້ → ບັນທຶກໃສ່ Google Drive ແທນ:
# from google.colab import drive; drive.mount('/content/drive')
# !cp model/laoasr-p8ft-int8.onnx /content/drive/MyDrive/

In [ ]:
# ຕົວເລືອກ: ຝາກຂຶ້ນ Hugging Face ແທນການດາວໂຫຼດ (ຕ້ອງມີ token)
# from huggingface_hub import login, upload_file
# login()
# upload_file(path_or_fileobj='model/laoasr-p8ft-int8.onnx',
#             path_in_repo='laoasr-p8ft-int8.onnx',
#             repo_id='<user>/laoasr-pruned', repo_type='model')

### ຫຼັງຈາກໄດ້ໄຟລ໌ແລ້ວ
ກັບມາທີ່ PC: ຄັດລອກ `laoasr-p8ft-int8.onnx` ໃສ່ `dev\laoasr\model\` ແລ້ວ
`dev\laoasr-pruned-car-test.ps1 -Tag p8ft` ເພື່ອວັດ CER/RTF/RSS ເທິງລົດ.